# Mini-Project 2: Uganda Tax Engine
**Objective**: Design an extensible, testable tax calculator modeling Uganda Revenue Authority (URA) regulations using strict Object-Oriented Programming (OOP) principles and `decimal.Decimal`.

In [1]:
# Setup and Imports
from decimal import Decimal
from src.tax_engine import (
    Money,
    TaxContext,
    PAYECalculator,
    PayrollPipeline,
    VATModule,
    CorporateIncomeTax,
    TaxPlanner,
    InvalidIncomeError
)
print("Tax engine modules imported successfully.")

Tax engine modules imported successfully.


## 1. Domain Model: Money Value Object & Decimal Safety
Floating-point numbers suffer from precision errors in financial math (e.g., `0.1 + 0.2 != 0.3`). The `Money` value object uses `decimal.Decimal` exclusively, implementing strict rounding rules (`ROUND_HALF_UP`) and operator overloading.

In [19]:
# Demonstrate Money Value Object operations
salary = Money(Decimal("5000000"))
bonus = Money(Decimal("1250000"))
total = salary + bonus
print(f"Salary: {salary}")
print(f"Bonus: {bonus}")
print(f"Total Compensation: {total}")
print(f"Is salary less than bonus? {salary < bonus}")

Salary: UGX 5,000,000.00
Bonus: UGX 1,250,000.00
Total Compensation: UGX 6,250,000.00
Is salary less than bonus? False


## 2. URA PAYE Calculator & Payroll Pipeline
Computing monthly resident PAYE across progressive bands, incorporating NSSF (5% employee, 10% employer), optional Local Service Tax (LST), and net pay.

In [5]:
# Run Payroll Pipeline for a sample gross monthly pay of UGX 4,500,000
pipeline = PayrollPipeline(lst_amount=Decimal("10000"))
payslip = pipeline.process_payroll(gross_pay=Decimal("4500000"), allowances=Decimal("500000"), is_resident=True)

for key, val in payslip.items():
    print(f"{key.replace('_', ' ').title()}: {val}")

Gross Pay: UGX 4,500,000.00
Allowances: UGX 500,000.00
Total Gross: UGX 5,000,000.00
Employee Nssf: UGX 225,000.00
Employer Nssf: UGX 450,000.00
Local Service Tax: UGX 10,000.00
Paye: UGX 1,317,750.00
Net Pay: UGX 3,447,250.00


### 3. VAT and Corporate Income Tax (CIT) Modules

In [21]:
# Test VAT Computation across mixed invoices
invoices = [
    {"amount": Decimal("10000000"), "type": "standard", "direction": "output"},
    {"amount": Decimal("4000000"), "type": "standard", "direction": "input"},
    {"amount": Decimal("2000000"), "type": "zero", "direction": "output"}
]
vat_result = VATModule.compute_net_vat(invoices)
print("VAT Summary:")
for k, v in vat_result.items():
    print(f"  {k}: {v}")

# Test Corporate Income Tax (CIT) at 30%[cite: 4]
cit_calc = CorporateIncomeTax()
cit_result = cit_calc.calculate(TaxContext(gross_income=Decimal("150000000"), allowances=Decimal("50000000")))
print(f"\nCorporate Income Tax Due: {cit_result.tax_due} (Effective Rate: {cit_result.effective_rate*100}%)")

VAT Summary:
  output_vat: UGX 1,800,000.00
  input_vat: UGX 720,000.00
  net_vat_payable: UGX 1,080,000.00

Corporate Income Tax Due: UGX 30,000,000.00 (Effective Rate: 30.00%)


## 4. Extension Task: TaxPlanner (Salary vs. Dividend Comparison)[cite: 4]

In [22]:
comparison = TaxPlanner.compare_salary_vs_dividend(annual_gross_salary=Decimal("120000000"), annual_dividend=Decimal("120000000"))
print("Tax Planner Scenario Analysis:")
for k, v in comparison.items():
    print(f"  {k}: {v}")

Tax Planner Scenario Analysis:
  salary_scenario_tax: UGX 34,659,000.00
  dividend_scenario_tax: UGX 12,000,000.00
  difference: UGX 22,659,000.00


## 5. Discussion Report & Design Justifications

### 1. Why is `float` unsafe for tax computation?
Standard floating-point numbers in Python (`float`) use binary IEEE 754 representation, which cannot precisely represent fractional decimal numbers like `0.1` or `0.2`. 
This introduces compounding rounding errors over multi-tiered progressive tax calculations. Using decimal.Decimal guarantees exact base-10 arithmetic required by tax regulations.

```python
# Concrete failing example with float:
print(0.1 + 0.2)  # Output: 0.30000000000000004
print(0.1 + 0.2 == 0.3)  # Output: False
```

### 2. Where Inheritance is Justified, and Where it was Deliberately Avoided?

**Inheritance** 
Inheritance is used for the abstract base class `Tax`, which defines the method `calculate(self, ctx: TaxContext) -> TaxResult`. Concrete tax engines, `PAYECalculator` and `CorporateIncomeTax`, implement this method. The shared base class enforces a unified polymorphic interface across different tax domains, so the pipeline can treat any tax engine the same way.

**Composition**
Composition is preferred in `PayrollPipeline` and `TaxPlanner`. These classes assemble complex logic by delegating work to distinct calculator modules, such as `PAYECalculator` and `Money`, rather than building deep class hierarchies. Each component can be tested, replaced, or changed independently, and the pipeline's behaviour is easier to follow because it is made of explicit collaborators.
